# 76 - Full LLM Judging of the Kerem/Manav External Review Sample

Notebook 73 found that only 2 of the 200 candidates Istari's reviewers labelled already had an internal LLM judge label, far too few to compute a meaningful LLM-human kappa. Asking Istari's annotators to label more data is not an option, so this notebook closes the gap from the other side instead: it judges all 200 candidates (the same two 100-candidate exports used in notebook 73) with the current LLM judge pair, Claude Sonnet 5 and Gemini 3.6 Flash, the same pair and enriched prompt used in notebook 66, then merges Kerem/Manav's labels, the author's blind labels (notebook 73), and the new LLM labels into one comparison file per query.

**Costs real money per API call.** 200 candidates x 2 judges = up to 400 calls. Run the `PILOT_N` cell first against a small slice to sanity-check cost and parsing before judging the full 200.

In [1]:
import os
import json
import time
import pandas as pd
import requests
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(override=True)
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

OUTPUT_DIR = Path("result/76_annotator_full_llm_judging")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = OUTPUT_DIR / "judge_cache.json"
MANUAL_REVIEW_PATH = OUTPUT_DIR / "manual_review_queue.json"

PILOT_N = None  # set to e.g. 6 for a cheap sanity-check slice across both queries before running the full 200

if GEMINI_API_KEY is None or ANTHROPIC_API_KEY is None:
    print("WARNING: missing GEMINI_API_KEY or ANTHROPIC_API_KEY in .env -- both judges are needed for agreement/disagreement voting.", flush=True)

## Part 1: Load Kerem/Manav's labels, the author's blind labels, and the corpus enrichment fields

Kerem/Manav's labels come straight from the two labeled review exports (the `relevance_label` column). The author's blind labels come from notebook 73's output file. Both are merged on `(query_id, domain)` rather than relying on row order, which is more robust than notebook 73's fixed-shuffle approach. The enrichment fields (state, district, municipality, organization_size, nace_code, summary_keywords) are pulled from `company_corpus.csv` since the review exports only carry a subset of fields, matching the enriched prompt notebook 66 already uses.

In [2]:
q1 = pd.read_excel("dataset/review_export_best100_q1_software_companies_labeled_.xlsx", sheet_name="review")
q1["query_id"] = 1
q1["query"] = "software companies"

q92 = pd.read_excel("dataset/review_export_best100_q92_wealth_management_firms_labeled_.xlsx", sheet_name="review")
q92["query_id"] = 92
q92["query"] = "wealth management firms"

annotator = pd.concat([q1, q92], ignore_index=True).rename(columns={"relevance_label": "annotator_label"})
print(f"Kerem/Manav rows: {len(annotator)}", flush=True)

marmee = pd.read_excel("result/73_annotator_match_kappa/annotator_match_sample.xlsx")
marmee = marmee[["query_id", "domain", "relevance_label"]].rename(columns={"relevance_label": "marmee_label"})
print(f"Marmee blind-label rows: {len(marmee)}", flush=True)

merged = annotator.merge(marmee, on=["query_id", "domain"], how="left")
assert merged["marmee_label"].notna().all(), "some candidates are missing an author blind label -- check notebook 73's output file"

corpus = pd.read_csv("dataset/company_corpus.csv")
enrich_cols = ["domain", "state", "district", "municipality", "organization_size", "nace_code", "summary_keywords"]
merged = merged.merge(corpus[enrich_cols].drop_duplicates("domain"), on="domain", how="left")

print(f"Merged + enriched rows: {len(merged)}", flush=True)
merged.head()

Kerem/Manav rows: 200
Marmee blind-label rows: 200
Merged + enriched rows: 200


,rank,domain,name,organization_type,country,summary,annotator_label,query_id,query,marmee_label,state,district,municipality,organization_size,nace_code,summary_keywords
0,3,suda3000.net.cn,厦门速达软件有限公司,Company,China,"Xiamen Suda Software Co., Ltd. is a software c...",2,1,software companies,2,NaN,NaN,NaN,Medium-sized (50-249),"NACE K: Telecommunication, computer programmin...","['Enterprise Resource Planning', 'Inventory Ma..."
1,4,hanyingsh.com,上海翰盈信息技术有限公司,Company,China,"Shanghai Hanying Information Technology Co., L...",2,1,software companies,2,Shanghai,Shanghai,NaN,Small (10-49),"NACE K: Telecommunication, computer programmin...","['enterprise informatization', 'software devel..."
2,5,grasp.com.cn,成都任我行软件股份有限公司,Company,China,"Chengdu Renwoxing Software Co., Ltd. is a soft...",2,1,software companies,2,NaN,NaN,NaN,Large enterprise (250+),"NACE K: Telecommunication, computer programmin...","['ERP', 'Inventory Management', 'Financial Man..."
3,8,ecaisoft.com,东莞市一彩软件有限公司,Company,China,"Dongguan Yicai Software Co., Ltd. is a softwar...",2,1,software companies,2,NaN,NaN,NaN,Small (10-49),"NACE K: Telecommunication, computer programmin...","['inventory management software', 'warehousing..."
4,9,sxyerp.com,东莞市数星软件科技有限公司,Company,China,"Dongguan Shuxing Software Technology Co., Ltd....",2,1,software companies,2,NaN,NaN,NaN,Small (10-49),"NACE K: Telecommunication, computer programmin...","['ERP software', 'Inventory management', 'Manu..."


## Part 2: Judge prompt and API calls

Identical to notebook 66's enriched prompt and judge functions (Claude Sonnet 5 + Gemini 3.6 Flash), copied verbatim rather than re-derived, so this round's labels are directly comparable to every other LLM-judged round in this thesis.

In [3]:
ENRICHED_JUDGE_PROMPT_TEMPLATE = """You are judging search result relevance for a company search engine.

Search query: "{query}"

Candidate company:
Name: {name}
Country: {country}
State/region: {state}
District: {district}
Municipality: {municipality}
Organization type: {organization_type}
Organization size: {organization_size}
NACE industry code: {nace_code}
Summary: {summary}
Summary keywords: {summary_keywords}

Rate how relevant this company is to the search query, using exactly one of these labels:
2 = highly relevant: a strong, direct match, the kind of company someone searching this query would expect and want to find.
1 = partially relevant: related to the query but not a strong direct match, e.g. an adjacent business, or a company that only partially fits the query's intent, or fits it for one line of business among several unrelated ones.
0 = not relevant: nothing meaningful to do with the query, including a company that merely mentions a query-related term in passing without it describing what the company actually does (for example, a venture capital firm whose portfolio includes software startups is NOT itself a software company for the query "software companies").

If the summary gives no verifiable content to judge from (placeholder text, clearly broken description), use 0 rather than guessing based on the name alone. Judge based on what the company currently does, not speculation about unstated subsidiaries or past business lines.

These are the same assessor guidelines given to the human reviewers labelling a parallel sample of this data, so your judgments can be meaningfully compared against theirs.

Respond with ONLY a JSON object: {{"label": <0, 1, or 2>, "reason": "<one short sentence>"}}"""


def build_prompt(row):
    fields = {c: (row.get(c, "") if pd.notna(row.get(c, "")) else "unknown") for c in
              ["query", "name", "country", "state", "district", "municipality",
               "organization_type", "organization_size", "nace_code", "summary", "summary_keywords"]}
    return ENRICHED_JUDGE_PROMPT_TEMPLATE.format(**fields)


def parse_judge_reply(text):
    try:
        start, end = text.index("{"), text.rindex("}") + 1
        parsed = json.loads(text[start:end])
        return int(parsed["label"]), parsed.get("reason", "")
    except (ValueError, KeyError, json.JSONDecodeError):
        return None, f"UNPARSEABLE: {text[:200]}"


def judge_gemini(prompt, max_retries=6):
    url = "https://generativelanguage.googleapis.com/v1beta/models/gemini-3.6-flash:generateContent"
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                url,
                headers={"x-goog-api-key": GEMINI_API_KEY, "Content-Type": "application/json"},
                json={"contents": [{"parts": [{"text": prompt}]}]},
                timeout=90,
            )
        except requests.exceptions.RequestException as e:
            if attempt == max_retries - 1:
                raise
            time.sleep(5 * (2 ** attempt))
            continue
        if resp.status_code == 429 or resp.status_code >= 500:
            if attempt == max_retries - 1:
                resp.raise_for_status()
            wait_s = int(resp.headers.get("Retry-After", 5 * (2 ** attempt)))
            time.sleep(wait_s)
            continue
        resp.raise_for_status()
        body = resp.json()
        try:
            text = body["candidates"][0]["content"]["parts"][0]["text"]
        except (KeyError, IndexError):
            return None, f"UNEXPECTED RESPONSE SHAPE: {json.dumps(body)[:300]}"
        return parse_judge_reply(text)
    return None, "EXHAUSTED RETRIES: repeated 429/5xx from Gemini"


def judge_claude_sonnet5(prompt):
    resp = requests.post(
        "https://api.anthropic.com/v1/messages",
        headers={"x-api-key": ANTHROPIC_API_KEY, "anthropic-version": "2023-06-01", "Content-Type": "application/json"},
        json={
            "model": "claude-sonnet-5", "max_tokens": 1024,
            "messages": [{"role": "user", "content": prompt}],
        },
        timeout=60,
    )
    resp.raise_for_status()
    content_blocks = resp.json()["content"]
    text_block = next((b["text"] for b in content_blocks if b.get("type") == "text"), None)
    if text_block is None:
        return None, f"NO TEXT BLOCK: {content_blocks}"
    return parse_judge_reply(text_block)

In [4]:
judge_df = merged.head(PILOT_N) if PILOT_N is not None else merged
print(f"{'PILOT MODE' if PILOT_N is not None else 'FULL RUN'}: judging {len(judge_df)} of {len(merged)} candidates", flush=True)

cache = json.load(open(CACHE_PATH)) if CACHE_PATH.exists() else {}
print(f"Loaded {len(cache)} cached entries", flush=True)

for i, row in judge_df.iterrows():
    key = f'{row["query_id"]}::{row["domain"]}'
    entry = cache.get(key, {})
    needs_gemini = entry.get("gemini", {}).get("label") is None
    needs_sonnet5 = entry.get("sonnet5", {}).get("label") is None

    if not (needs_gemini or needs_sonnet5):
        continue  # both judges already cached for this candidate -- no API call, no disk write, no sleep needed

    prompt = build_prompt(row)

    if needs_gemini:
        try:
            label, reason = judge_gemini(prompt)
            entry["gemini"] = {"label": label, "reason": reason}
        except (requests.exceptions.RequestException, KeyError, IndexError) as e:
            print(f"  [gemini] error on {row['domain']}: {e}", flush=True)
    if needs_sonnet5:
        try:
            label, reason = judge_claude_sonnet5(prompt)
            entry["sonnet5"] = {"label": label, "reason": reason}
        except (requests.exceptions.RequestException, KeyError, IndexError) as e:
            print(f"  [sonnet5] error on {row['domain']}: {e}", flush=True)

    cache[key] = entry
    tmp_path = CACHE_PATH.with_suffix(".json.tmp")
    json.dump(cache, open(tmp_path, "w"), indent=2, default=str)
    tmp_path.replace(CACHE_PATH)  # atomic save after every single candidate -- paid calls, never redo work

    if (i + 1) % 20 == 0 or (i + 1) == len(judge_df):
        n_done = sum(1 for e in cache.values() if e.get("gemini", {}).get("label") is not None and e.get("sonnet5", {}).get("label") is not None)
        print(f"  row {i+1}/{len(judge_df)} reached, {n_done} candidates fully judged so far", flush=True)
    time.sleep(2.0)

print("Done judging.", flush=True)

FULL RUN: judging 200 of 200 candidates
Loaded 0 cached entries
  row 20/200 reached, 20 candidates fully judged so far
  row 40/200 reached, 40 candidates fully judged so far
  row 60/200 reached, 60 candidates fully judged so far
  row 80/200 reached, 80 candidates fully judged so far
  row 100/200 reached, 100 candidates fully judged so far
  row 120/200 reached, 120 candidates fully judged so far
  row 140/200 reached, 140 candidates fully judged so far
  row 160/200 reached, 160 candidates fully judged so far
  row 180/200 reached, 180 candidates fully judged so far
  row 200/200 reached, 200 candidates fully judged so far
Done judging.


## Part 3: Resolve LLM labels and build the four-way comparison

Where Gemini and Sonnet 5 agree, that label is accepted directly (`llm_label`). Where they disagree, the candidate is flagged for the same manual-review process used everywhere else in this thesis rather than silently resolved, and `llm_label` is left blank so it is not mistaken for a settled value.

In [5]:
def resolve_row(row):
    key = f'{row["query_id"]}::{row["domain"]}'
    entry = cache.get(key, {})
    gemini_label = entry.get("gemini", {}).get("label")
    sonnet_label = entry.get("sonnet5", {}).get("label")
    if gemini_label is None or sonnet_label is None:
        return pd.Series({"gemini_label": gemini_label, "sonnet5_label": sonnet_label, "llm_label": None, "llm_agreement": "not judged"})
    if gemini_label == sonnet_label:
        return pd.Series({"gemini_label": gemini_label, "sonnet5_label": sonnet_label, "llm_label": gemini_label, "llm_agreement": "unanimous"})
    return pd.Series({"gemini_label": gemini_label, "sonnet5_label": sonnet_label, "llm_label": None, "llm_agreement": "disagreement"})

resolved = merged.join(merged.apply(resolve_row, axis=1))

manual_review = resolved[resolved["llm_agreement"] == "disagreement"][
    ["query_id", "query", "domain", "name", "summary", "gemini_label", "sonnet5_label"]
]
manual_review.to_json(MANUAL_REVIEW_PATH, orient="records", indent=2)

n_judged = resolved["llm_agreement"].isin(["unanimous", "disagreement"]).sum()
print(f"Judged: {n_judged}/{len(resolved)}", flush=True)
print(resolved["llm_agreement"].value_counts(), flush=True)
print(f"Manual tie-breaks needed: {len(manual_review)} -- see {MANUAL_REVIEW_PATH}", flush=True)

Judged: 200/200
llm_agreement
unanimous       199
disagreement      1
Name: count, dtype: int64
Manual tie-breaks needed: 1 -- see result/76_annotator_full_llm_judging/manual_review_queue.json


## Part 4: Save the four-way comparison (Kerem/Manav, Marmee blind, Gemini, Sonnet 5)

Saves one combined CSV/JSON across both queries, plus a per-query Excel file matching the original review-export format with the extra label columns added, for easy manual reading alongside the original exports.

In [6]:
display_cols = ["query_id", "query", "domain", "name", "organization_type", "country", "summary",
                "annotator_label", "marmee_label", "gemini_label", "sonnet5_label", "llm_label", "llm_agreement"]
final = resolved[display_cols].copy()

final.to_csv(OUTPUT_DIR / "four_way_comparison.csv", index=False)
final.to_json(OUTPUT_DIR / "four_way_comparison.json", orient="records", indent=2)

for qid, qname in [(1, "q1_software_companies"), (92, "q92_wealth_management_firms")]:
    out_path = OUTPUT_DIR / f"four_way_comparison_{qname}.xlsx"
    final[final["query_id"] == qid].to_excel(out_path, index=False, sheet_name="comparison")
    print(f"Saved {out_path}", flush=True)

print(f"\nSaved {OUTPUT_DIR / 'four_way_comparison.csv'} and .json", flush=True)
final.head()

Saved result/76_annotator_full_llm_judging/four_way_comparison_q1_software_companies.xlsx
Saved result/76_annotator_full_llm_judging/four_way_comparison_q92_wealth_management_firms.xlsx

Saved result/76_annotator_full_llm_judging/four_way_comparison.csv and .json


,query_id,query,domain,name,organization_type,country,summary,annotator_label,marmee_label,gemini_label,sonnet5_label,llm_label,llm_agreement
0,1,software companies,suda3000.net.cn,厦门速达软件有限公司,Company,China,"Xiamen Suda Software Co., Ltd. is a software c...",2,2,2,2,2.0,unanimous
1,1,software companies,hanyingsh.com,上海翰盈信息技术有限公司,Company,China,"Shanghai Hanying Information Technology Co., L...",2,2,2,2,2.0,unanimous
2,1,software companies,grasp.com.cn,成都任我行软件股份有限公司,Company,China,"Chengdu Renwoxing Software Co., Ltd. is a soft...",2,2,2,2,2.0,unanimous
3,1,software companies,ecaisoft.com,东莞市一彩软件有限公司,Company,China,"Dongguan Yicai Software Co., Ltd. is a softwar...",2,2,2,2,2.0,unanimous
4,1,software companies,sxyerp.com,东莞市数星软件科技有限公司,Company,China,"Dongguan Shuxing Software Technology Co., Ltd....",2,2,2,2,2.0,unanimous


## Part 5: Headline kappa figures (LLM ensemble vs. Kerem/Manav, and vs. Marmee)

Restricted to the rows where the LLM ensemble reached unanimous agreement (`llm_label` is not null), matching how every other LLM-human comparison in this thesis is computed.

In [7]:
from sklearn.metrics import cohen_kappa_score

judged = final[final["llm_label"].notna()].copy()
judged["llm_label"] = judged["llm_label"].astype(int)
print(f"LLM-unanimous rows available for kappa: {len(judged)}/{len(final)}", flush=True)

for label_a, name_a, label_b, name_b in [
    ("llm_label", "LLM ensemble", "annotator_label", "Kerem/Manav"),
    ("llm_label", "LLM ensemble", "marmee_label", "Marmee (blind)"),
]:
    a, b = judged[label_a], judged[label_b]
    agreement = (a == b).mean()
    kappa = cohen_kappa_score(a, b)
    a_bin, b_bin = (a >= 1).astype(int), (b >= 1).astype(int)
    kappa_bin = cohen_kappa_score(a_bin, b_bin)
    print(f"{name_a} vs {name_b}: n={len(judged)}, raw agreement={agreement:.1%}, kappa(3-class)={kappa:.3f}, kappa(binary)={kappa_bin:.3f}", flush=True)

summary = {
    "n_total": int(len(final)),
    "n_llm_unanimous": int(len(judged)),
    "n_llm_disagreement": int((final["llm_agreement"] == "disagreement").sum()),
}
with open(OUTPUT_DIR / "kappa_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
summary

LLM-unanimous rows available for kappa: 199/200
LLM ensemble vs Kerem/Manav: n=199, raw agreement=92.5%, kappa(3-class)=0.196, kappa(binary)=nan
LLM ensemble vs Marmee (blind): n=199, raw agreement=94.5%, kappa(3-class)=0.069, kappa(binary)=0.000


/home/ma/ma_ma/ma_mpandya/Thesis/thesis/lib64/python3.12/site-packages/sklearn/metrics/_classification.py:534: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/ma/ma_ma/ma_mpandya/Thesis/thesis/lib64/python3.12/site-packages/sklearn/metrics/_classification.py:897: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


{'n_total': 200, 'n_llm_unanimous': 199, 'n_llm_disagreement': 1}